# 01 — Data Extraction
**NST DVA Capstone 2 | B_G6_DIGITOMICS**

**Sector:** EdTech / Digital Behaviour Analytics  
**Dataset:** Student Digital Behaviour Data (`student_digital_behaviour_data.csv`)  
**Notebook Purpose:** Load the raw dataset from `data/raw/`, inspect its structure, document initial observations, and confirm it meets the capstone requirements before passing it to the cleaning pipeline.


## 1. Import Libraries

In [2]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

Libraries loaded successfully.


## 2. Define File Paths

In [3]:
# ── Adjust this path if running from a different working directory ──
RAW_DATA_PATH = '../data/raw/student_digital_behaviour_data.csv'

# Verify the file exists before loading
if os.path.exists(RAW_DATA_PATH):
    print(f'✅ File found: {RAW_DATA_PATH}')
else:
    print(f'❌ File NOT found at: {RAW_DATA_PATH}')
    print('   → Make sure the CSV is placed in data/raw/ before running this notebook.')

✅ File found: ../data/raw/student_digital_behaviour_data.csv


## 3. Load Raw Dataset

In [4]:
df_raw = pd.read_csv(RAW_DATA_PATH)

print(f'Rows    : {df_raw.shape[0]:,}')
print(f'Columns : {df_raw.shape[1]}')
print()
print('Column names:')
for i, col in enumerate(df_raw.columns, 1):
    print(f'  {i:>2}. {col}')

Rows    : 500,000
Columns : 48

Column names:
   1. student_id
   2. country
   3. development_level
   4. poverty_rate_percent
   5. internet_infrastructure_index
   6. average_internet_speed_mbps
   7. age
   8. gender
   9. urban_rural
  10. family_income_level
  11. device_access
  12. internet_access_hours
  13. education_level
  14. field_of_study
  15. academic_motivation
  16. online_learning_hours
  17. social_media_hours
  18. sessions_per_day
  19. average_session_length_minutes
  20. late_night_usage
  21. education_content_hours
  22. short_video_hours
  23. entertainment_content_hours
  24. news_content_hours
  25. likes_given_per_day
  26. comments_written_per_day
  27. posts_created_per_week
  28. late_night_score
  29. brain_rot_index
  30. brain_rot_level
  31. attention_span_minutes
  32. study_hours_per_week
  33. class_attendance_rate
  34. productivity_score
  35. sleep_hours
  36. stress_level
  37. anxiety_score
  38. depression_score
  39. ads_viewed_per_day
  

## 4. Preview First 5 Rows

In [5]:
df_raw.head()

,student_id,country,development_level,poverty_rate_percent,internet_infrastructure_index,average_internet_speed_mbps,age,gender,urban_rural,family_income_level,...,ads_viewed_per_day,ads_clicked_per_week,impulse_purchase_score,digital_spending_per_month,cyberbullying_exposure,adult_content_exposure,digital_addiction_score,wellbeing_index,academic_risk_score,financial_risk_score
0,1,Qatar,Developing,16.30,54.93,27.19,21,Male,Rural,High,...,38.699484,3.119569,3.681066,79.508194,No,No,8.179932,66.662166,0.0,26.516722
1,2,USA,Developed,8.75,94.39,85.34,25,Male,Urban,Middle,...,157.400429,18.358090,6.538867,73.452464,No,Yes,22.073122,43.892278,0.0,39.257741
2,3,Mexico,Developing,23.64,47.24,73.55,18,Female,Urban,Low,...,79.603536,11.758299,5.150660,35.753069,No,No,12.591830,65.484625,0.0,47.542155
3,4,Canada,Developed,14.51,90.50,188.19,25,Male,Urban,Middle,...,69.318324,7.906197,3.195383,47.607487,No,Yes,8.008238,57.909392,0.0,23.436122
4,5,Sri Lanka,Underdeveloped,62.28,36.84,11.02,15,Other,Rural,Middle,...,144.019899,19.427839,7.180234,82.941313,No,No,21.551334,53.686356,0.0,47.308493


## 5. Data Types & Memory Usage

In [6]:
df_raw.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 48 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   student_id                      500000 non-null  int64  
 1   country                         500000 non-null  str    
 2   development_level               500000 non-null  str    
 3   poverty_rate_percent            500000 non-null  float64
 4   internet_infrastructure_index   500000 non-null  float64
 5   average_internet_speed_mbps     500000 non-null  float64
 6   age                             500000 non-null  int64  
 7   gender                          500000 non-null  str    
 8   urban_rural                     500000 non-null  str    
 9   family_income_level             500000 non-null  str    
 10  device_access                   500000 non-null  str    
 11  internet_access_hours           500000 non-null  float64
 12  education_level            

## 6. Missing Value Audit

In [7]:
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

missing_report = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

if missing_report.empty:
    print('No missing values detected at this stage.')
else:
    print(f'Columns with missing values ({len(missing_report)} found):\n')
    print(missing_report.to_string())

Columns with missing values (2 found):

                 Missing Count  Missing %
field_of_study          247349      49.47
brain_rot_level           6262       1.25


## 7. Duplicate Row Check

In [8]:
dup_count = df_raw.duplicated().sum()
print(f'Fully duplicate rows : {dup_count}')

# Also check for duplicate student IDs specifically
if 'student_id' in df_raw.columns:
    dup_ids = df_raw['student_id'].duplicated().sum()
    print(f'Duplicate student_id : {dup_ids}')

Fully duplicate rows : 0
Duplicate student_id : 0


## 8. Categorical Columns — Unique Value Inspection

In [9]:
# Identify likely categorical columns (object dtype or low-cardinality)
cat_cols = df_raw.select_dtypes(include='object').columns.tolist()

print(f'Categorical columns detected: {len(cat_cols)}\n')
for col in cat_cols:
    unique_vals = df_raw[col].dropna().unique()
    print(f'  {col} ({len(unique_vals)} unique):')
    print(f'    → {sorted(unique_vals[:20])}')  # show first 20 to flag inconsistencies
    print()

Categorical columns detected: 12

  country (50 unique):
    → ['Bangladesh', 'Canada', 'China', 'Denmark', 'France', 'Germany', 'India', 'Japan', 'Kuwait', 'Mexico', 'Peru', 'Poland', 'Qatar', 'Saudi Arabia', 'South Africa', 'Sri Lanka', 'Sweden', 'Thailand', 'USA', 'Vietnam']

  development_level (3 unique):
    → ['Developed', 'Developing', 'Underdeveloped']

  gender (3 unique):
    → ['Female', 'Male', 'Other']

  urban_rural (2 unique):
    → ['Rural', 'Urban']

  family_income_level (3 unique):
    → ['High', 'Low', 'Middle']

  device_access (4 unique):
    → ['Both', 'Laptop', 'Shared Device', 'Smartphone']

  education_level (6 unique):
    → ['Diploma', 'Dropout', 'Graduate', 'PhD', 'Postgraduate', 'School']

  field_of_study (6 unique):
    → ['Arts', 'Business', 'Law', 'Medicine', 'STEM', 'Social Sciences']

  late_night_usage (4 unique):
    → ['Always', 'Never', 'Often', 'Sometimes']

  brain_rot_level (2 unique):
    → ['Low', 'Moderate']

  cyberbullying_exposure (2 un

## 9. Numeric Columns — Descriptive Statistics

In [10]:
df_raw.describe(include=[np.number]).T

,count,mean,std,min,25%,50%,75%,max
student_id,500000.0,250000.500000,144337.711635,1.000000,125000.750000,250000.500000,375000.250000,500000.000000
poverty_rate_percent,500000.0,26.157521,15.246496,5.210000,13.660000,22.330000,37.930000,62.280000
internet_infrastructure_index,500000.0,60.211633,20.732890,20.760000,43.130000,55.750000,78.920000,94.390000
average_internet_speed_mbps,500000.0,72.663154,65.315300,5.740000,20.420000,45.630000,107.410000,238.050000
age,500000.0,19.996474,3.161021,15.000000,17.000000,20.000000,23.000000,25.000000
internet_access_hours,500000.0,5.013043,1.581660,0.500000,3.911805,5.012280,6.112144,11.273316
academic_motivation,500000.0,5.390749,1.741491,1.000000,4.238213,5.457743,6.611136,10.000000
online_learning_hours,500000.0,6.318560,1.848903,0.000000,5.080963,6.354281,7.585492,13.530034
social_media_hours,500000.0,3.360787,1.292885,0.000000,2.457786,3.290051,4.196469,9.000000
sessions_per_day,500000.0,9.901511,3.382135,1.000000,7.547886,9.744129,12.097976,26.032059


## 10. Capstone Requirements Check

In [11]:
rows, cols = df_raw.shape
non_id_cols = [c for c in df_raw.columns if c not in ['student_id']]

print('=' * 50)
print('CAPSTONE REQUIREMENTS VALIDATION')
print('=' * 50)
print(f'Row count         : {rows:,}  → {"✅ PASS (≥5,000)" if rows >= 5000 else "❌ FAIL (need ≥5,000)"}')
print(f'Column count      : {cols}   → {"✅ PASS (≥8 columns)" if cols >= 8 else "❌ FAIL (need ≥8)"}')
print(f'Has missing vals  : {df_raw.isnull().any().any()} → {"✅ Realistic issues present" if df_raw.isnull().any().any() else "⚠️  No missing values — verify raw source"}')
print(f'Numeric columns   : {len(df_raw.select_dtypes(include=np.number).columns)}')
print(f'Categorical cols  : {len(df_raw.select_dtypes(include="object").columns)}')
print('=' * 50)

CAPSTONE REQUIREMENTS VALIDATION
Row count         : 500,000  → ✅ PASS (≥5,000)
Column count      : 48   → ✅ PASS (≥8 columns)
Has missing vals  : True → ✅ Realistic issues present
Numeric columns   : 36
Categorical cols  : 12


## 11. Initial Observations (ETL Lead Notes)

Document your raw-data findings here before cleaning begins. Update after running cells above.

| # | Observation | Column(s) Affected | Action in 02_cleaning |
|---|------------|--------------------|-----------------------|
| 1 | `field_of_study` is NaN for school/dropout students | `field_of_study` | Impute with `'Not Applicable'` |
| 2 | `late_night_usage` is string ('Sometimes', 'Often', 'Never') | `late_night_usage` | Encode to ordinal int |
| 3 | `gender` has 'Other' category | `gender` | Keep as-is — valid value |
| 4 | `device_access` has 'Shared Device' — may limit usage | `device_access` | Keep, flag in analysis |
| 5 | `brain_rot_level` is categorical ('Low', 'Medium', 'High') | `brain_rot_level` | Label-encode for ML use |
| 6 | `cyberbullying_exposure` / `adult_content_exposure` are Yes/No | both | Binary encode (0/1) |
| *Add more as you discover them* | | | |

> **ETL Lead:** Fill in any additional observations from cells above before proceeding to `02_cleaning.ipynb`.

## 12. Save Extraction Snapshot (Optional Diagnostic)

Saves a small sample for quick visual inspection without modifying the raw file.

In [12]:
# Save a 50-row head snapshot for quick reference — does NOT replace raw data
snapshot_path = '../data/raw/extraction_snapshot_50rows.csv'
df_raw.head(50).to_csv(snapshot_path, index=False)
print(f'Snapshot saved → {snapshot_path}')
print()
print('✅ Extraction notebook complete.')
print('   → Next step: Run 02_cleaning.ipynb')

Snapshot saved → ../data/raw/extraction_snapshot_50rows.csv

✅ Extraction notebook complete.
   → Next step: Run 02_cleaning.ipynb
